# Event-study difference-in-differences (simulated)

Dr. Pavanam Thomas  
Copyright 2026 Dr. Pavanam Thomas

Full write-up: `CASE_STUDY.md`. This notebook imports `econci`; it does not re-implement the estimators.

All samples are simulated. Conventional TWFE is not automatically valid under staggered adoption with heterogeneous effects. The educational group-time ATT below is not a Callaway–Sant'Anna estimator.

Problem → formalization → assumptions → computation/estimation → validation → interpretation → limitations.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if (root / "src").exists():
    sys.path.insert(0, str(root / "src"))
elif (root.parent / "src").exists():
    sys.path.insert(0, str(root.parent / "src"))

from econci import dgp, did, plots

## 2x2 under parallel trends (by construction)

Estimand: ATT. Identifying variation: differential change relative to the never-treated group. Assumption: parallel trends in $Y(0)$.

In [ ]:
two = dgp.simulate_did_2x2(n_treat=150, n_control=150, att=2.0, parallel_trends=True, seed=42)
fit = did.fit_did_2x2(two)
print("Interaction (ATT under PT):", float(fit.params["_did"]), "SE", float(fit.bse["_did"]))
print("DiD of means:", did.did_att_from_means(two))

bad = dgp.simulate_did_2x2(n_treat=150, n_control=150, att=2.0, parallel_trends=False, trend_gap=1.2, seed=42)
fit_bad = did.fit_did_2x2(bad)
print("Same estimator when PT is false:", float(fit_bad.params["_did"]))

## Event study and pre-treatment coefficients

Static `treated` is 0 in pre-adoption periods for units that will adopt. That coding is required; it is tested in `tests/test_did.py`.

In [ ]:
ev = dgp.simulate_event_study(n_treat=80, n_control=80, n_periods=10, treat_period=6, att=2.0, seed=42)
design = did.build_event_study_design(ev)
pre = design[(design["ever_treated"] == 1) & (design["period"] < design["g"])]
print("Pre-adoption treated dummy all zero:", bool((pre["treated"] == 0).all()))
names = list(design.attrs["event_dummy_names"])
results, coefs = did.fit_event_study(design, dummy_names=names)
print(did.joint_pretrend_test(results, did.lead_names_from_dummies(names)))
plot_df = did.event_study_coef_table_with_omitted(coefs)
plots.plot_event_study(plot_df, "outputs/figures/notebook_event_study.png")
plot_df

## Staggered adoption: TWFE versus known cohort ATTs

Educational group-time ATT: simple never-treated contrasts. Not Callaway–Sant'Anna (no not-yet-treated comparison, no doubly robust score, no CS inference).

In [ ]:
stag = dgp.simulate_staggered_heterogeneous(att_early=4.0, att_late=-1.5, seed=42)
twfe = did.fit_twfe_static(stag)
att_gt = did.group_time_att_educational(stag)
agg = did.aggregate_group_time_att(att_gt)
print("TWFE static treated:", float(twfe.params["treated"]))
print("Educational group-time simple ATT:", agg["att_simple"])
print("Known early / late ATTs: 4.0 / -1.5")
att_gt.head()

## What cannot be concluded

This notebook does not estimate a real policy. A quiet pre-trend test does not prove parallel trends after adoption. TWFE in the staggered DGP is not a group-time ATT. See the five-way distinction at the end of `CASE_STUDY.md` (estimated effect; identifying assumption; diagnostic evidence; uncertainty; what cannot be concluded).